# step5 통제 — 전 층 스윕 (RQ3 인과)

**어느 스텝·어느 RQ:** step5(지침 인과, RQ3). 이미 돌린 통제를 **전 층으로 다시** 잰다.

**무엇을 확인하나.**
step5는 처치(반대 지침 지시어 덮기)를 **전 층**에서 쟀는데, 통제는 **봉우리 층 한 곳**에서만
돌았다. 그래서 순효과(처치 − 통제)를 낼 수 있는 층이 하나뿐이고, **층별 순효과 곡선을
만들 수 없다.** step3에는 있는 그림이 step5에는 없는 비대칭이다.

이번 실행은 **같은 통제 조건을 층 하나가 아니라 전 층에서** 잰다. 프롬프트·공여·시드는
그대로고, 바뀌는 것은 `layers='sweep'` 하나뿐이다.

**지금 있는 것과 없는 것**

| | 층 | 개수 |
|---|---|---|
| 처치 `opposite_instruction` | 전 층 (28~36) | 336 |
| 통제 `self`·`unrelated_word` (기존) | **1개 층** | 672 |
| **통제 전 층 (이번에 돌릴 것)** | **전 층** | **672** |

**결과가 어느 쪽으로 나오든 무슨 뜻인지 미리 밝힌다.**

| 나오는 그림 | 뜻 |
|---|---|
| 순효과 봉우리가 **지금 층과 같다** | 지금 보고한 step5 숫자가 그대로 맞다. 층별 곡선이 새로 붙는다 |
| 순효과 봉우리가 **다른 층이다** | 봉우리를 놓치고 있었다는 뜻이라 **순효과 크기는 지금보다 커진다.** 방향·부호는 안 바뀐다. 그 경우 바뀐 값으로 다시 보고한다 |
| 통제가 전 층에서 **0 근처** | 거품이 봉우리 층에만 있었다는 뜻. 처치 곡선을 거의 그대로 읽어도 된다 |
| 통제가 전 층에서 **크다** | 거품이 층 전체에 깔려 있다. 층별 곡선이 처치 곡선과 모양이 달라진다 |

**미리 적어두는 예측.** step3에서는 네 모델 모두 **원값 봉우리 = 순효과 봉우리**였다
(L25·L20·L15·L18). step5도 같을 것으로 본다. 즉 봉우리 층은 그대로고 곡선만 새로 생길 것이다.

**부하.** 모델당 42묶음 × 방향 2 × 공여 2 = **168조건**, 조건마다 전 층 스윕.
처치 스윕 실측(조건당 8~15초)으로 잡으면 **모델 하나에 20~40분**, 네 모델 합쳐 **약 2시간**.

> ⚠️ **Llama는 짝이 적게 남는다.** 지침이 행동을 거의 못 흔든 조건(판정 불가)이 많아,
> 기존 통제에서도 168조건 중 32개만 짝이 맞았다. 전 층으로 바꿔도 이건 달라지지 않는다 —
> Llama 곡선은 다른 세 모델보다 신뢰구간이 넓게 나온다.

> ⚠️ **메모리.** DeepSeek-Coder-6.7B는 float16으로 약 13.5GB라 T4(15GB)에서 빠듯하다.
> 처치 스윕은 같은 조건에서 돌아갔지만, 끊기면 셀 ⑤를 다시 실행하면 된다(이미 저장된 조건은 건너뛴다).

> ⚠️ **Colab 기본 환경을 그대로 씁니다.** 저장된 결과는 전부 Colab 기본값
> (torch 2.11 · transformers 5.13.1 · Python 3.12)에서 나왔습니다. `requirements.txt` 의
> 고정 버전을 설치하면 오히려 깨집니다. 셀 ③이 기존 결과의 `meta` 와 지금 환경을
> 대조해, 다르면 멈춥니다 — 이 실행분은 기존 결과에서 **빼는 값**이라 환경이 갈리면 안 됩니다.

> ⚠️ **저장 폴더가 기존 통제와 다르다** — `results/step5_instr-cause-control-sweep/`.
> 같은 폴더에 두면 집계가 `(모델·묶음·방향·공여)`로만 묶기 때문에 층이 다른 두 실행분이
> 조용히 서로를 덮는다. 폴더를 갈라 그 사고를 막는다(CLAUDE.md §6·§7).

In [ ]:
# ② 환경 — GPU 확인, 무작위값 42 고정
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)

In [ ]:
# ③ 저장소 클론 및 브랜치 체크아웃
import os, sys
# 절대경로로 고정한다 — 상대경로면 이미 저장소 안에 들어와 있을 때 중첩 클론된다
if not os.path.isdir('/content/HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git /content/HCLT_2026
%cd /content/HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
sys.path.insert(0, 'src')
print('브랜치:', BRANCH)

# ── 환경 대조 ─────────────────────────────────────────────────────────────
# 이 실행분은 **기존 처치 결과에서 빼는 값**이 된다. 두 실행의 환경이 갈리면 그 차이가
# 표기 효과인지 환경 차이인지 가를 수 없다. 그래서 기존 통제 파일이 기록해 둔 환경과
# 지금 환경을 맞춰 본다(하드코딩하지 않고 결과 파일에서 읽는다).
import glob, json as _json, transformers
ref = None
for f in sorted(glob.glob('results/step5_instr-cause-control/*.json'))[:1]:
    ref = _json.load(open(f)).get('meta') or None

now = {'torch': torch.__version__, 'transformers': transformers.__version__,
       'python': sys.version.split()[0]}
print()
print('환경 대조 — 기존 통제 결과가 기록한 값과 지금 값')
print(f"{'':<14}{'기존':<22}{'지금':<22}")
drift = []
for k in ('torch', 'transformers', 'python'):
    old = (ref or {}).get(k, '(기록 없음)')
    print(f'{k:<14}{old:<22}{now[k]:<22}')
    if ref and old != now[k]:
        drift.append(k)

ALLOW_VERSION_DRIFT = False      # ← 환경이 달라도 강행하려면 True 로 바꾼다
if drift and not ALLOW_VERSION_DRIFT:
    raise RuntimeError(
        '환경이 기존 통제 실행과 다릅니다: ' + ', '.join(drift) + '\n'
        '이 실행분은 기존 처치 결과에서 빼는 값이라, 환경이 갈리면 차이의 해석이 흐려집니다.\n'
        '\n가장 빠른 해결: **런타임 > 연결 해제 및 런타임 삭제** 후 새 런타임에서 셀 ① 부터.\n'
        '새 VM 은 Colab 기본값이라 그대로 통과합니다.\n'
        '\n환경이 다른 줄 알면서 강행하려면 이 셀의 ALLOW_VERSION_DRIFT 를 True 로 두십시오. '
        '실제 환경은 결과 JSON 의 meta 에 자동 기록되므로 사후 확인이 가능합니다.')
print('환경 일치 — 진행합니다' if ref else '기존 meta 가 없어 대조를 건너뜁니다')

In [ ]:
# ④ 조건 설정 — 모델 하나, 전 층 스윕, 통제 공여 2종
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
print('이번 모델:', MODEL.family)

BLOCKS = list(range(42))
DONORS = ['self', 'unrelated_word']        # 통제 2종 (처치 opposite는 이미 전 층으로 돌렸다)

conditions = []
for block in BLOCKS:
    pre = PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL, pool_block=block)
    for notation in (Notation.CAMEL, Notation.SNAKE):
        for donor in DONORS:
            conditions.append(Condition(
                model=MODEL, preceding=pre,
                instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=notation),
                intervention=Intervention(
                    kind=InterventionKind.KEY_VALUE,
                    layers='sweep',            # ← 여기 하나만 기존 통제와 다르다 (전에는 층 1개)
                    donor=donor, target='instruction',
                    # 처치와 방식을 맞춰야 같은 자로 뺄 수 있다
                    kinds=('key', 'value', 'key_value')),
                seed=SEED, token_unit='mean'))

print('조건 수:', len(conditions), f'(묶음 {len(BLOCKS)} × 방향 2 × 통제 {len(DONORS)})')
print('조건마다 전 층을 훑고, 층마다 방식 3가지를 잰다.')
print('예상 시간: 모델 하나에 20~40분')

In [ ]:
# ⑤ 실행 — 조건마다 즉시 저장(재개). 이미 있는 조건은 건너뛴다
import time
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np

STEP = 'step5_instr-cause-control-sweep'
PREDICTION = ('통제이므로 층 전체에서 넘어감이 0 근처여야 정상. '
              '순효과 봉우리는 기존 봉우리 층과 같을 것으로 예측한다(step3 근거).')

todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)
    print(f'  층수 {handle.num_layers}')
    seen, t0 = [], time.time()
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle)              # 전 층 스윕
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ3', prediction=PREDICTION))
        ex = out.metrics.extra
        vals = [v.get('value__recovery') for v in out.metrics.per_layer.values()
                if v.get('value__recovery') is not None]
        if vals and not ex.get('undecidable'):
            seen.append(max(vals, key=abs))      # 그 조건에서 가장 크게 움직인 층의 값
        if i % 10 == 0 or i == len(todo):
            el = time.time() - t0
            print(f'  [{i}/{len(todo)}] 잰 층 {len(out.metrics.per_layer)}개 · '
                  f'공여={ex["donor"]:<24} '
                  f'지금까지 넘어감(내용만, 층 최대) 평균 {np.mean(seen) if seen else float("nan"):.3f} '
                  f'(0에 가까워야 정상) · '
                  f'{el/60:.0f}분 경과, 남은 예상 {el/i*(len(todo)-i)/60:.0f}분')
    del handle
    import gc; gc.collect(); torch.cuda.empty_cache()
print('완료')

In [ ]:
# ⑥ 결과 불러오기 + 처치와 층 수가 맞는지 확인
import glob, json as _json
from harness import result_path
from harness.results import load_result

recs = [load_result(result_path(c, step=STEP)) for c in conditions
        if result_path(c, step=STEP).exists()]
print('불러온 조건:', len(recs), '/', len(conditions))
if not recs:
    raise SystemExit('아직 결과가 없습니다 — 셀 ⑤ 를 먼저 돌리십시오')

n_layers = {len(r.metrics.per_layer) for r in recs}
print('조건당 층 수:', sorted(n_layers))
print('판정 불가로 빠질 조건:', sum(1 for r in recs if r.metrics.extra.get('undecidable')),
      '/', len(recs))

# 처치가 몇 층을 쟀는지 읽어와 맞춰 본다. 다르면 층별 순효과 곡선이 부분만 나온다.
t_layers = set()
for f in glob.glob('results/step5_instr-cause/*.json'):
    d = _json.load(open(f))
    if d['condition']['model']['family'] == MODEL.family:
        t_layers.add(len(d['metrics']['per_layer']))
print('처치의 층 수:', sorted(t_layers))
if n_layers != t_layers:
    raise RuntimeError(
        f'층 수가 처치와 다릅니다 — 통제 {sorted(n_layers)} vs 처치 {sorted(t_layers)}.\n'
        '층별 순효과 곡선을 그리면 일부 층에서만 빼기가 되어 곡선이 끊깁니다.')
print('층 수 일치 — 전 층에서 빼기가 됩니다')

In [ ]:
# ⑦ 요약 — 통제가 층마다 얼마인지, 그래서 순효과 봉우리가 어디인지
import glob, json as _json
from collections import defaultdict
import numpy as np

KINDS = ['value', 'key', 'key_value']
KO = {'value': '내용만', 'key': '어텐션만', 'key_value': '둘다'}

# (1) 이번에 돌린 통제 — 층별
ctrl = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))   # 공여→방식→층→{묶음·방향: 값}
for r in recs:
    ex = r.metrics.extra
    if ex.get('undecidable'):
        continue
    key = (r.condition.preceding.pool_block, r.condition.instruction.target_notation.value)
    for L, vals in r.metrics.per_layer.items():
        for k in KINDS:
            v = vals.get(f'{k}__recovery')
            if v is not None:
                ctrl[ex['donor']][k][int(L)][key] = v

# (2) 처치 — step5 본실험에서 같은 모델을 읽어 온다
treat = defaultdict(lambda: defaultdict(dict))
for f in glob.glob('results/step5_instr-cause/*.json'):
    d = _json.load(open(f))
    if d['condition']['model']['family'] != MODEL.family:
        continue
    ex = d['metrics']['extra']
    if ex.get('undecidable') or abs(ex['S_clean'] - ex['S_base']) < 1.0:
        continue
    key = (d['condition']['preceding']['pool_block'],
           d['condition']['instruction']['target_notation'])
    for L, vals in d['metrics']['per_layer'].items():
        for k in KINDS:
            v = vals.get(f'{k}__recovery')
            if v is not None:
                treat[k][int(L)][key] = v

# (3) 순효과 = 처치 − 통제, 묶음끼리 짝지어 뺀다
print(f'[{MODEL.family}] 층별 순효과 (자기 통제 기준, 내용만)')
print(f"{'층':>5}{'처치':>10}{'통제':>10}{'순효과':>10}{'짝 수':>8}")
best, best_v = None, float('-inf')
for L in sorted(treat['value']):
    t, c = treat['value'][L], ctrl['control_self']['value'].get(L, {})
    keys = set(t) & set(c)
    if not keys:
        continue
    d = [t[k] - c[k] for k in keys]
    if np.mean(d) > best_v:
        best, best_v = L, float(np.mean(d))
    if L % 4 == 0 or L == max(treat['value']):
        print(f'{L:>5}{np.mean([t[k] for k in keys]):>10.3f}'
              f'{np.mean([c[k] for k in keys]):>10.3f}{np.mean(d):>10.3f}{len(keys):>8}')

print()
print(f'순효과 봉우리 층: L{best}  (값 {best_v:.3f})')
PREV_PEAK = {'qwen': 27, 'deepseek': 17, 'llama': 24, 'stability': 19}[MODEL.family]
print(f'기존에 통제를 걸었던 층: L{PREV_PEAK}  (처치 원값 곡선의 봉우리)')
print('→ 같으면 지금까지 보고한 값이 그대로 맞다. 다르면 순효과 크기가 커지고, 그 값으로 다시 보고한다.')

print()
print('전체 집계는 저장소 스크립트로 한다(손으로 만든 수치는 남기지 않는다):')
print('  python scripts/step5_net_effect.py results/step5_instr-cause results/step5_instr-cause-control-sweep')
print('  python scripts/step5_figures.py')

In [ ]:
# ⑦-2 결과 폴더를 zip으로 묶어 내려받기
import shutil
from google.colab import files
name = f'step5_control_sweep_{MODEL.family}'
shutil.make_archive(f'/content/{name}', 'zip', 'results/step5_instr-cause-control-sweep')
print('묶었다:', f'/content/{name}.zip')
files.download(f'/content/{name}.zip')